# LeetCode 905 — Sort Array By Parity bằng Python

**Link đề:** https://leetcode.com/problems/sort-array-by-parity/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers, Sorting

```python
class Solution:
    def sortArrayByParity(self, nums: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Kiểm tra chẵn/lẻ bằng `% 2`, và mẹo bit `x & 1`.
2. Cách dễ nhất: tách hai list **chẵn** và **lẻ** rồi nối; cách Pythonic: `sorted(nums, key=lambda x: x % 2)`.
3. Kỹ thuật **phân hoạch tại chỗ** (in-place partition) bằng **hai con trỏ**: (a) hai đầu tiến vào giữa, (b) con trỏ ghi (`write`) quét một chiều.
4. Hiểu **tính ổn định** (stable) và **hoán đổi** `a[i], a[j] = a[j], a[i]`.
5. Trace, chứng minh invariant, so sánh 5 cách và kiểm thử bằng **hàm kiểm tra tính hợp lệ** (vì đề chấp nhận nhiều đáp án).

## 1. Đề bài và ví dụ

Cho mảng số nguyên `nums`, hãy đưa **mọi số chẵn lên đầu**, theo sau là **mọi số lẻ**. Trả về **bất kỳ** mảng nào thỏa điều kiện.

| nums | Một output hợp lệ | Các output khác cũng đúng |
|---|---|---|
| `[3,1,2,4]` | `[2,4,3,1]` | `[4,2,3,1]`, `[2,4,1,3]`, `[4,2,1,3]` |
| `[0]` | `[0]` | — |

**Constraints:** `1 <= nums.length <= 5000`, `0 <= nums[i] <= 5000`.

Điều kiện đúng: tồn tại vị trí `k` sao cho `nums[0..k-1]` toàn chẵn và `nums[k..]` toàn lẻ. Thứ tự **bên trong** mỗi nhóm không quan trọng. Lưu ý `0` là số **chẵn**.

## 2. Nền tảng Python cần biết

### 2.1. Chẵn/lẻ bằng `% 2`

`x % 2` là **phần dư** khi chia cho 2: `0` nếu chẵn, `1` nếu lẻ (với `x ≥ 0`). Mẹo bit: `x & 1` lấy **bit cuối** của `x` — cũng là `0`/`1` cho chẵn/lẻ.

In [1]:
for x in [0, 1, 2, 7, 10]:
    print(f"x={x:>2}: x % 2 = {x % 2} | x & 1 = {x & 1} | bin = {bin(x)} -> {'chẵn' if x % 2 == 0 else 'lẻ'}")

x= 0: x % 2 = 0 | x & 1 = 0 | bin = 0b0 -> chẵn
x= 1: x % 2 = 1 | x & 1 = 1 | bin = 0b1 -> lẻ
x= 2: x % 2 = 0 | x & 1 = 0 | bin = 0b10 -> chẵn
x= 7: x % 2 = 1 | x & 1 = 1 | bin = 0b111 -> lẻ
x=10: x % 2 = 0 | x & 1 = 0 | bin = 0b1010 -> chẵn


### 2.2. Hàm kiểm tra kết quả hợp lệ

Vì có nhiều đáp án, ta không so với một output cố định mà kiểm 2 điều:
1. Kết quả là một **hoán vị** của input (cùng các phần tử, cùng số lần) — so `sorted(a) == sorted(b)`.
2. **Không có số lẻ nào đứng trước số chẵn** — tức dãy `x % 2` không giảm: `0,0,...,0,1,...,1`.

In [2]:
def is_valid(original: list[int], result: list[int]) -> bool:
    if sorted(original) != sorted(result):
        return False
    parities = [x % 2 for x in result]
    return all(p <= q for p, q in zip(parities, parities[1:]))

print(is_valid([3, 1, 2, 4], [2, 4, 3, 1]))   # True
print(is_valid([3, 1, 2, 4], [4, 2, 1, 3]))   # True
print(is_valid([3, 1, 2, 4], [2, 3, 4, 1]))   # False: 3 lẻ đứng trước 4 chẵn
print(is_valid([3, 1, 2, 4], [2, 4, 1, 1]))   # False: không phải hoán vị

True
True
False
False


### 2.3. Hoán đổi hai phần tử: `a[i], a[j] = a[j], a[i]`

Python tính **toàn bộ vế phải trước** (tạo tuple `(a[j], a[i])`), rồi mới gán lần lượt. Nhờ vậy không cần biến tạm.

In [3]:
a = [10, 20, 30]
a[0], a[2] = a[2], a[0]
print(a)

[30, 20, 10]


### 2.4. List comprehension có điều kiện và nối list

`[x for x in nums if x % 2 == 0]` lọc số chẵn. `list1 + list2` nối thành list mới.

In [4]:
nums = [3, 1, 2, 4]
evens = [x for x in nums if x % 2 == 0]
odds = [x for x in nums if x % 2 == 1]
print("evens =", evens, "| odds =", odds, "| nối =", evens + odds)

evens = [2, 4] | odds = [3, 1] | nối = [2, 4, 3, 1]


## 3. Cách 1 — Tách hai list rồi nối

Đây là cách tự nhiên nhất: "gom chẵn một chỗ, lẻ một chỗ, rồi ghép".

In [5]:
def sort_parity_two_lists(nums: list[int]) -> list[int]:
    evens = [x for x in nums if x % 2 == 0]
    odds = [x for x in nums if x % 2 == 1]
    return evens + odds

print(sort_parity_two_lists([3, 1, 2, 4]))
print(sort_parity_two_lists([0]))

[2, 4, 3, 1]
[0]


- Thời gian `O(n)` (2 lượt lọc), bộ nhớ `O(n)` cho hai list mới.
- **Ổn định** (stable): các số chẵn giữ nguyên thứ tự tương đối như input, các số lẻ cũng vậy.

## 4. Cách 2 — `sorted` với `key`

`sorted(nums, key=f)` sắp xếp theo giá trị `f(x)` thay vì `x`. Với `key = x % 2`: số chẵn có key `0` đứng trước số lẻ có key `1`.

`lambda x: x % 2` là **hàm vô danh** (anonymous function) một dòng: nhận `x`, trả `x % 2`.

Thuật toán sắp xếp của Python (Timsort) **ổn định** → kết quả giống hệt cách 1.

In [6]:
square = lambda x: x * x
print("lambda:", square(5))

def sort_parity_sorted(nums: list[int]) -> list[int]:
    return sorted(nums, key=lambda x: x % 2)

print(sort_parity_sorted([3, 1, 2, 4]))
print("key của từng phần tử:", [(x, x % 2) for x in [3, 1, 2, 4]])

lambda: 25
[2, 4, 3, 1]
key của từng phần tử: [(3, 1), (1, 1), (2, 0), (4, 0)]


### 4.1. Nút thắt của hai cách trên

- Cách 2 tốn `O(n log n)` cho một bài chỉ cần **phân hai nhóm**.
- Cả hai đều tạo mảng **mới** `O(n)`. Phỏng vấn thường hỏi tiếp: *"Làm **tại chỗ** (in-place), bộ nhớ `O(1)` được không?"*

**Insight:** ta không cần sắp xếp đầy đủ, chỉ cần **phân hoạch** (partition) thành hai vùng. Phân hoạch tại chỗ = dùng **hai con trỏ** và **hoán đổi**.

## 5. Cách 3 — Hai con trỏ từ hai đầu (in-place)

- `left` bắt đầu từ đầu mảng, tìm số **lẻ** (đang "nằm sai chỗ" ở vùng đầu).
- `right` bắt đầu từ cuối mảng, tìm số **chẵn** (đang "nằm sai chỗ" ở vùng cuối).
- Khi cả hai cùng dừng ở phần tử sai chỗ → **hoán đổi** chúng.
- Lặp đến khi `left >= right`.

In [7]:
def sort_parity_two_ends(nums: list[int]) -> list[int]:
    left, right = 0, len(nums) - 1
    while left < right:
        if nums[left] % 2 == 0:          # đã đúng chỗ (chẵn ở đầu)
            left += 1
        elif nums[right] % 2 == 1:       # đã đúng chỗ (lẻ ở cuối)
            right -= 1
        else:                            # left lẻ, right chẵn -> đổi
            nums[left], nums[right] = nums[right], nums[left]
            left += 1
            right -= 1
    return nums

print(sort_parity_two_ends([3, 1, 2, 4]))
print(sort_parity_two_ends([1, 3, 5, 2, 4, 6]))

[4, 2, 1, 3]
[6, 4, 2, 5, 3, 1]


Chú ý: hàm này **sửa trực tiếp** list truyền vào (vì list là đối tượng có thể thay đổi — mutable — và được truyền theo tham chiếu). Khi test, nên truyền **bản sao** `nums[:]` hoặc `list(nums)` nếu muốn giữ input.

In [8]:
original = [3, 1, 2, 4]
result = sort_parity_two_ends(original)
print("result is original?", result is original, "| original bây giờ:", original)
original = [3, 1, 2, 4]
result = sort_parity_two_ends(original[:])
print("Dùng bản sao -> original giữ nguyên:", original, "| result:", result)

result is original? True | original bây giờ: [4, 2, 1, 3]
Dùng bản sao -> original giữ nguyên: [3, 1, 2, 4] | result: [4, 2, 1, 3]


## 6. Cách 4 — Lời giải chính: con trỏ ghi `write` (quét một chiều)

Ý tưởng giống "dọn bàn": `write` là vị trí **tiếp theo** để đặt một số chẵn.
- Duyệt `i` từ trái sang phải.
- Gặp số chẵn `nums[i]` → hoán đổi nó về vị trí `write`, rồi `write += 1`.

Sau vòng lặp, `nums[0..write-1]` toàn chẵn, phần còn lại toàn lẻ.

In [9]:
def sort_parity_write(nums: list[int]) -> list[int]:
    write = 0
    for i in range(len(nums)):
        if nums[i] % 2 == 0:
            nums[write], nums[i] = nums[i], nums[write]
            write += 1
    return nums

print(sort_parity_write([3, 1, 2, 4]))
print(sort_parity_write([1, 3, 5, 2, 4, 6]))

[2, 4, 3, 1]
[2, 4, 6, 1, 3, 5]


Cách 4 giữ **thứ tự tương đối của số chẵn** (chúng được đặt vào lần lượt), nhưng **không** giữ thứ tự số lẻ. Cách 3 không giữ thứ tự nhóm nào cả. Đề không yêu cầu ổn định nên cả hai đều đúng.

## 7. Cách 5 — Ghi vào mảng mới từ hai đầu (một lượt, `O(n)` nhớ)

Tạo `res` cùng độ dài. Số chẵn ghi từ **đầu** (`lo` tăng), số lẻ ghi từ **cuối** (`hi` giảm). Một lượt duyệt, không hoán đổi.

In [10]:
def sort_parity_fill(nums: list[int]) -> list[int]:
    res = [0] * len(nums)
    lo, hi = 0, len(nums) - 1
    for x in nums:
        if x % 2 == 0:
            res[lo] = x; lo += 1
        else:
            res[hi] = x; hi -= 1
    return res

print(sort_parity_fill([3, 1, 2, 4]))

[2, 4, 1, 3]


`[0] * n` tạo list gồm `n` số 0 — chuẩn bị sẵn chỗ để ghi theo chỉ số.

## 8. TRACE từng bước

### 8.1. Bảng trace cách 4 (`write`) với `nums = [3, 1, 2, 4]`

| i | nums[i] | Chẵn? | Hành động | nums sau | write sau |
|---:|---:|:---:|---|---|---:|
| 0 | 3 | không | — | [3,1,2,4] | 0 |
| 1 | 1 | không | — | [3,1,2,4] | 0 |
| 2 | 2 | **có** | đổi nums[0]↔nums[2] | [2,1,3,4] | 1 |
| 3 | 4 | **có** | đổi nums[1]↔nums[3] | [2,4,3,1] | 2 |

Kết quả `[2,4,3,1]` — trùng output mẫu của đề.

In [11]:
def trace_write(nums):
    nums = nums[:]
    write = 0
    print(f"Bắt đầu: {nums}, write={write}")
    for i in range(len(nums)):
        if nums[i] % 2 == 0:
            print(f"  i={i}: {nums[i]} chẵn -> đổi nums[{write}]={nums[write]} với nums[{i}]={nums[i]}", end="")
            nums[write], nums[i] = nums[i], nums[write]
            write += 1
            print(f" -> {nums}, write={write}")
        else:
            print(f"  i={i}: {nums[i]} lẻ  -> bỏ qua                       {nums}, write={write}")
    print("Kết quả:", nums, "| vùng chẵn = nums[:write] =", nums[:write])
    return nums

_ = trace_write([3, 1, 2, 4])

Bắt đầu: [3, 1, 2, 4], write=0
  i=0: 3 lẻ  -> bỏ qua                       [3, 1, 2, 4], write=0
  i=1: 1 lẻ  -> bỏ qua                       [3, 1, 2, 4], write=0
  i=2: 2 chẵn -> đổi nums[0]=3 với nums[2]=2 -> [2, 1, 3, 4], write=1
  i=3: 4 chẵn -> đổi nums[1]=1 với nums[3]=4 -> [2, 4, 3, 1], write=2
Kết quả: [2, 4, 3, 1] | vùng chẵn = nums[:write] = [2, 4]


### 8.2. Trace cách 3 (hai đầu) — ví dụ có nhiều lần đổi

In [12]:
def trace_two_ends(nums):
    nums = nums[:]
    left, right = 0, len(nums) - 1
    print("Bắt đầu:", nums)
    while left < right:
        if nums[left] % 2 == 0:
            print(f"  left={left} ({nums[left]} chẵn, đúng chỗ) -> left+1")
            left += 1
        elif nums[right] % 2 == 1:
            print(f"  right={right} ({nums[right]} lẻ, đúng chỗ) -> right-1")
            right -= 1
        else:
            nums[left], nums[right] = nums[right], nums[left]
            print(f"  ĐỔI left={left} và right={right} -> {nums}")
            left += 1; right -= 1
    print("Kết quả:", nums)
    return nums

_ = trace_two_ends([1, 3, 5, 2, 4, 6])

Bắt đầu: [1, 3, 5, 2, 4, 6]
  ĐỔI left=0 và right=5 -> [6, 3, 5, 2, 4, 1]
  ĐỔI left=1 và right=4 -> [6, 4, 5, 2, 3, 1]
  ĐỔI left=2 và right=3 -> [6, 4, 2, 5, 3, 1]
Kết quả: [6, 4, 2, 5, 3, 1]


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def sortArrayByParity(self, nums: list[int]) -> list[int]:
        write = 0
        for i in range(len(nums)):
            if nums[i] % 2 == 0:
                nums[write], nums[i] = nums[i], nums[write]
                write += 1
        return nums

print(Solution().sortArrayByParity([3, 1, 2, 4]), Solution().sortArrayByParity([0]))

[2, 4, 3, 1] [0]


### 9.1. Các bản nộp khác (đều Accepted)

In [14]:
class SolutionSorted:
    def sortArrayByParity(self, nums: list[int]) -> list[int]:
        return sorted(nums, key=lambda x: x & 1)

class SolutionTwoEnds:
    def sortArrayByParity(self, nums: list[int]) -> list[int]:
        left, right = 0, len(nums) - 1
        while left < right:
            if nums[left] % 2 == 0:
                left += 1
            elif nums[right] % 2 == 1:
                right -= 1
            else:
                nums[left], nums[right] = nums[right], nums[left]
                left += 1
                right -= 1
        return nums

print(SolutionSorted().sortArrayByParity([3, 1, 2, 4]), SolutionTwoEnds().sortArrayByParity([3, 1, 2, 4]))

[2, 4, 3, 1] [4, 2, 1, 3]


## 10. Vì sao đúng?

**Cách 4 (write) — invariant:** trước khi xét chỉ số `i`:
- `nums[0 .. write-1]` toàn **chẵn**,
- `nums[write .. i-1]` toàn **lẻ**,
- `nums[i ..]` chưa xét; mảng luôn là hoán vị của input (chỉ dùng hoán đổi).

Khởi tạo: `write = i = 0`, hai vùng rỗng ✔. Bước `i`: nếu `nums[i]` lẻ → vùng lẻ mở rộng thêm 1 ✔. Nếu chẵn → đổi với `nums[write]` (là số lẻ đầu vùng lẻ, hoặc chính `nums[i]` nếu vùng lẻ rỗng) → vùng chẵn thêm 1, vùng lẻ "trượt" sang phải 1 ✔. Kết thúc `i = n`: chẵn trước, lẻ sau. ∎

**Cách 3 (hai đầu) — invariant:** `nums[0..left-1]` toàn chẵn và `nums[right+1..]` toàn lẻ. Mỗi vòng hoặc mở rộng một vùng, hoặc đổi một cặp (lẻ, chẵn) sai chỗ rồi mở rộng cả hai. `right - left` giảm mỗi vòng → dừng. Khi `left >= right`, phần giữa có tối đa 1 phần tử, chẵn hay lẻ đều hợp lệ ở ranh giới. ∎

## 11. Độ phức tạp và so sánh

`n = len(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | In-place? | Ổn định? |
|---|---|---|---|:---:|---|
| 1. Hai list | lọc chẵn + lẻ | `O(n)` | `O(n)` | không | cả hai nhóm |
| 2. `sorted` key | Timsort theo `x % 2` | `O(n log n)` | `O(n)` | không | cả hai nhóm |
| 3. Hai đầu | đổi cặp sai chỗ | `O(n)` | `O(1)` | **có** | không |
| 4. Con trỏ `write` | đẩy chẵn về trước | `O(n)` | `O(1)` | **có** | chỉ nhóm chẵn |
| 5. Điền hai đầu | mảng mới | `O(n)` | `O(n)` | không | chẵn ổn định, lẻ bị đảo |

Cách 3 thường **ít hoán đổi** hơn (mỗi phần tử bị đổi tối đa một lần); cách 4 có thể đổi nhiều lần nhưng code đơn giản và mở rộng tốt cho bài 283 Move Zeroes.

In [15]:
def count_swaps_write(nums):
    nums = nums[:]; write = swaps = 0
    for i in range(len(nums)):
        if nums[i] % 2 == 0:
            if write != i:
                swaps += 1
            nums[write], nums[i] = nums[i], nums[write]
            write += 1
    return swaps

def count_swaps_two_ends(nums):
    nums = nums[:]; l, r = 0, len(nums) - 1; swaps = 0
    while l < r:
        if nums[l] % 2 == 0: l += 1
        elif nums[r] % 2 == 1: r -= 1
        else:
            nums[l], nums[r] = nums[r], nums[l]; swaps += 1; l += 1; r -= 1
    return swaps

sample = [1, 2, 4, 6, 8]
print("Số lần đổi thật sự - write:", count_swaps_write(sample), "| hai đầu:", count_swaps_two_ends(sample))

Số lần đổi thật sự - write: 4 | hai đầu: 1


## 12. Bộ test: mẫu đề, biên, đối chiếu bằng `is_valid`

In [16]:
methods = [sort_parity_two_lists, sort_parity_sorted, sort_parity_two_ends, sort_parity_write, sort_parity_fill,
           Solution().sortArrayByParity, SolutionSorted().sortArrayByParity, SolutionTwoEnds().sortArrayByParity]

cases = [
    ("đề VD1", [3, 1, 2, 4]),
    ("đề VD2", [0]),
    ("toàn chẵn", [2, 4, 6, 0]),
    ("toàn lẻ", [1, 3, 5]),
    ("đã đúng", [2, 4, 1, 3]),
    ("ngược hoàn toàn", [1, 3, 5, 2, 4, 6]),
    ("có 0 và trùng", [0, 1, 0, 1, 1, 0]),
    ("2 phần tử sai chỗ", [1, 2]),
]
for name, nums in cases:
    for f in methods:
        res = f(nums[:])                     # truyền bản sao vì vài cách sửa tại chỗ
        assert is_valid(nums, res), (name, f, res)
    print(f"PASS  {name}: {nums} -> {Solution().sortArrayByParity(nums[:])}")

PASS  đề VD1: [3, 1, 2, 4] -> [2, 4, 3, 1]
PASS  đề VD2: [0] -> [0]
PASS  toàn chẵn: [2, 4, 6, 0] -> [2, 4, 6, 0]
PASS  toàn lẻ: [1, 3, 5] -> [1, 3, 5]
PASS  đã đúng: [2, 4, 1, 3] -> [2, 4, 1, 3]
PASS  ngược hoàn toàn: [1, 3, 5, 2, 4, 6] -> [2, 4, 6, 1, 3, 5]
PASS  có 0 và trùng: [0, 1, 0, 1, 1, 0] -> [0, 0, 0, 1, 1, 1]
PASS  2 phần tử sai chỗ: [1, 2] -> [2, 1]


In [17]:
import random
random.seed(905)
for _ in range(1000):
    nums = [random.randint(0, 20) for _ in range(random.randint(1, 12))]
    for f in methods:
        assert is_valid(nums, f(nums[:]))
# tính ổn định của cách 1 / 2 (so với nhau phải giống hệt)
for _ in range(200):
    nums = [random.randint(0, 20) for _ in range(random.randint(1, 12))]
    assert sort_parity_two_lists(nums) == sort_parity_sorted(nums)
print("1000 mảng ngẫu nhiên: 8 cách đều hợp lệ; cách 1 và 2 cho kết quả giống hệt (cùng ổn định).")

1000 mảng ngẫu nhiên: 8 cách đều hợp lệ; cách 1 và 2 cho kết quả giống hệt (cùng ổn định).


## 13. Lỗi người mới hay gặp

### Lỗi 1: Gặp số lẻ ở `left` là đổi ngay, không kiểm `nums[right]`, rồi vẫn tăng `left`
Phần tử vừa được đổi về vị trí `left` có thể lại là **số lẻ** (vì `nums[right]` lẻ) nhưng ta đã bước qua nó → số lẻ nằm lại ở vùng đầu.

In [18]:
def wrong_always_swap(nums):
    left, right = 0, len(nums) - 1
    while left < right:
        if nums[left] % 2 == 1:          # gặp lẻ là đổi luôn, không xem right là gì
            nums[left], nums[right] = nums[right], nums[left]
            right -= 1
        left += 1                        # SAI: sau khi đổi, phần tử mới ở left chưa được kiểm
    return nums

bad = wrong_always_swap([1, 2, 3])
print("Sai:", bad, "| hợp lệ?", is_valid([1, 2, 3], bad))

Sai: [3, 2, 1] | hợp lệ? False


### Lỗi 2: Xóa phần tử khỏi list trong khi đang duyệt nó
```python
for x in nums:
    if x % 2: nums.remove(x); nums.append(x)   # SAI
```
Sửa list trong lúc `for` đang chạy làm vòng lặp **bỏ qua** phần tử và có thể không dừng như mong muốn.

In [19]:
def wrong_remove_while_iterating(nums):
    nums = nums[:]
    count = 0
    for x in nums:
        count += 1
        if count > 20:                     # chặn để demo không chạy mãi
            break
        if x % 2 == 1:
            nums.remove(x)
            nums.append(x)
    return nums

bad = wrong_remove_while_iterating([1, 3, 2, 5, 4])
print("Sai:", bad, "| hợp lệ?", is_valid([1, 3, 2, 5, 4], bad))

Sai: [3, 2, 4, 5, 1] | hợp lệ? False


### Lỗi 3: Dùng `x % 2 == 1` để kiểm chẵn (ngược điều kiện) → số lẻ lên trước.

### Lỗi 4: Coi `0` là "không chẵn không lẻ"
`0 % 2 == 0` → `0` là chẵn, phải ở nhóm đầu.

### Lỗi 5: Quên rằng hàm in-place sửa input, rồi dùng lại input đã bị thay đổi để test
Luôn test với `nums[:]`.

In [20]:
data = [3, 1, 2, 4]
first = sort_parity_write(data)
print("Lần 1:", first, "| data giờ cũng là:", data, "<- input đã bị sửa!")
print("0 là chẵn?", 0 % 2 == 0)

Lần 1: [2, 4, 3, 1] | data giờ cũng là: [2, 4, 3, 1] <- input đã bị sửa!
0 là chẵn? True


## 14. Mẹo nhận dạng pattern

- "Đưa các phần tử thỏa điều kiện **lên đầu / ra cuối**", "phân hai nhóm", "không cần giữ thứ tự" → **partition bằng hai con trỏ**.
- Cần giữ thứ tự của nhóm được đưa lên → **con trỏ `write`** (giống Move Zeroes).
- Cho phép mảng mới và cần ổn định cả hai nhóm → **lọc hai list** hoặc `sorted(key=...)`.
- Ba nhóm (ví dụ 0/1/2) → **Dutch National Flag** (ba con trỏ).

Bài cùng họ: 283 Move Zeroes, 922 Sort Array By Parity II, 75 Sort Colors, 27 Remove Element, 2149 Rearrange Array Elements by Sign.

## 15. Biến thể

### 15.1. LeetCode 922 — Sort Array By Parity II
Mảng có số chẵn và lẻ **bằng nhau**; cần `nums[i]` chẵn khi `i` chẵn, lẻ khi `i` lẻ. Dùng hai con trỏ: `i` đi các vị trí chẵn, `j` đi các vị trí lẻ.

In [21]:
def sort_parity_ii(nums: list[int]) -> list[int]:
    j = 1
    for i in range(0, len(nums), 2):          # vị trí chẵn
        if nums[i] % 2 == 1:                  # sai chỗ -> tìm số chẵn ở vị trí lẻ
            while nums[j] % 2 == 1:
                j += 2
            nums[i], nums[j] = nums[j], nums[i]
    return nums

print(sort_parity_ii([4, 2, 5, 7]))

[4, 5, 2, 7]


`range(0, len(nums), 2)` — tham số thứ ba là **bước nhảy**: `0, 2, 4, ...`.

### 15.2. Ba nhóm: chia 3 dư 0, 1, 2 (Dutch National Flag)

In [22]:
def sort_by_mod3(nums: list[int]) -> list[int]:
    lo, mid, hi = 0, 0, len(nums) - 1
    while mid <= hi:
        r = nums[mid] % 3
        if r == 0:
            nums[lo], nums[mid] = nums[mid], nums[lo]; lo += 1; mid += 1
        elif r == 1:
            mid += 1
        else:
            nums[mid], nums[hi] = nums[hi], nums[mid]; hi -= 1
    return nums

print(sort_by_mod3([5, 3, 1, 6, 4, 2, 9, 7, 8]))

[3, 6, 9, 7, 4, 1, 2, 8, 5]


## 16. Bài tập tự luyện

Hàm TODO chỉ định nghĩa, không được gọi khi chạy notebook.

### Bài 1 — Tính tay
Trace cách `write` với `[2, 3, 4, 5, 6]`. Liệt kê `nums` và `write` sau mỗi bước.

### Bài 2 — Dự đoán output
```python
sorted([5, 2, 7, 4], key=lambda x: x % 2)
sorted([5, 2, 7, 4], key=lambda x: (x % 2, x))
```

### Bài 3 — Điền chỗ trống: cách `write`

In [23]:
def sort_parity_exercise(nums):
    write = 0
    for i in range(len(nums)):
        # TODO: nếu nums[i] chẵn -> đổi với nums[write], tăng write
        pass
    return nums

# assert is_valid([3, 1, 2, 4], sort_parity_exercise([3, 1, 2, 4]))

### Bài 4 — Tự viết cách hai đầu

In [24]:
def sort_parity_two_ends_exercise(nums):
    # TODO
    pass

# assert is_valid([1, 3, 5, 2, 4, 6], sort_parity_two_ends_exercise([1, 3, 5, 2, 4, 6]))

### Bài 5 — Biến thể: Move Zeroes (LeetCode 283)
Đưa mọi số `0` ra **cuối**, giữ nguyên thứ tự các số khác 0, làm tại chỗ.

In [25]:
def move_zeroes(nums) -> None:
    # TODO: con trỏ write cho số khác 0
    pass

# a = [0, 1, 0, 3, 12]; move_zeroes(a); assert a == [1, 3, 12, 0, 0]

### Bài 6 — Mini challenge: chẵn tăng dần ở đầu, lẻ **giảm dần** ở cuối
Ví dụ `[3, 1, 2, 4]` → `[2, 4, 3, 1]`. Viết bằng một lời gọi `sorted` với `key` phù hợp.

In [26]:
def evens_asc_odds_desc(nums):
    # TODO: key là một tuple
    pass

# assert evens_asc_odds_desc([3, 1, 2, 4]) == [2, 4, 3, 1]
# assert evens_asc_odds_desc([5, 8, 7, 6, 1]) == [6, 8, 7, 5, 1]

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | 2 chẵn ở đầu | đổi với chính nó khi write == i | `[2,4,3,5,6]` rồi `[2,4,6,5,3]` |
| 2 | key quyết định thứ tự | tuple so sánh từng phần | `[2,4,5,7]`, `[2,4,5,7]` |
| 3 | `if nums[i] % 2 == 0` | hoán đổi bằng tuple | `write += 1` |
| 4 | left tìm lẻ, right tìm chẵn | if / elif / else | đổi rồi dịch cả hai |
| 5 | giống bài 3 | điều kiện `nums[i] != 0` | đổi `nums[write], nums[i]` |
| 6 | tuple key | lẻ giảm dần = `-x` | `key=lambda x: (x % 2, x if x % 2 == 0 else -x)` |

## 18. Đáp án tham khảo

**Bài 1:** i=0 (2 chẵn, đổi với chính nó) → `[2,3,4,5,6]`, write=1; i=1 (3 lẻ); i=2 (4 chẵn, đổi nums[1]↔nums[2]) → `[2,4,3,5,6]`, write=2; i=3 (5 lẻ); i=4 (6 chẵn, đổi nums[2]↔nums[4]) → `[2,4,6,5,3]`, write=3.

**Bài 2:** `[2, 4, 5, 7]` (ổn định: 2 trước 4, 5 trước 7) và `[2, 4, 5, 7]`.

In [27]:
def ans_sort_parity(nums):
    write = 0
    for i in range(len(nums)):
        if nums[i] % 2 == 0:
            nums[write], nums[i] = nums[i], nums[write]
            write += 1
    return nums

def ans_two_ends(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        if nums[l] % 2 == 0:
            l += 1
        elif nums[r] % 2 == 1:
            r -= 1
        else:
            nums[l], nums[r] = nums[r], nums[l]
            l += 1; r -= 1
    return nums

def ans_move_zeroes(nums):
    write = 0
    for i in range(len(nums)):
        if nums[i] != 0:
            nums[write], nums[i] = nums[i], nums[write]
            write += 1

def ans_evens_asc_odds_desc(nums):
    return sorted(nums, key=lambda x: (x % 2, x if x % 2 == 0 else -x))

assert ans_sort_parity([2, 3, 4, 5, 6]) == [2, 4, 6, 5, 3]
assert sorted([5, 2, 7, 4], key=lambda x: x % 2) == [2, 4, 5, 7]
assert sorted([5, 2, 7, 4], key=lambda x: (x % 2, x)) == [2, 4, 5, 7]
assert is_valid([3, 1, 2, 4], ans_sort_parity([3, 1, 2, 4]))
assert is_valid([1, 3, 5, 2, 4, 6], ans_two_ends([1, 3, 5, 2, 4, 6]))
a = [0, 1, 0, 3, 12]; ans_move_zeroes(a); assert a == [1, 3, 12, 0, 0]
assert ans_evens_asc_odds_desc([3, 1, 2, 4]) == [2, 4, 3, 1]
assert ans_evens_asc_odds_desc([5, 8, 7, 6, 1]) == [6, 8, 7, 5, 1]
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh và checklist submit

```python
class Solution:
    def sortArrayByParity(self, nums):
        write = 0                                   # chỗ đặt số chẵn tiếp theo
        for i in range(len(nums)):
            if nums[i] % 2 == 0:                    # gặp chẵn
                nums[write], nums[i] = nums[i], nums[write]
                write += 1
        return nums                                 # [chẵn ... | lẻ ...]
```

**Câu nhớ:** *"Phân hai nhóm không cần sắp xếp: một con trỏ ghi, một con trỏ đọc."*

**Checklist:**
- [ ] Điều kiện chẵn là `x % 2 == 0` (và `0` là chẵn)?
- [ ] Hoán đổi bằng tuple, cập nhật `write` sau khi đổi?
- [ ] Nếu dùng hai đầu: dừng khi `left >= right`, không đổi khi đã đúng chỗ?
- [ ] Test toàn chẵn, toàn lẻ, 1 phần tử?
- [ ] Khi tự test hàm in-place: truyền bản sao `nums[:]`, kiểm bằng `is_valid`?